# Задача 2: Предобработка и редактирование текста

Пайплайн очистки данных для классификации обращений в приёмную комиссию.

**Этапы:**
1. Настройка окружения.
2. Демонстрация функций на одном примере.
3. Полный пайплайн (очистка + лемматизация + стемминг + сленг).
4. Сплит train/test + аугментация только train.
5. Честная оценка F1: с аугментацией vs без.

In [1]:
# ==========================================
# БЛОК 1: НАСТРОЙКА ОКРУЖЕНИЯ
# ==========================================
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from src.preprocessing import (
    clean_text,
    fix_typos,
    lemmatize_text,
    replace_slang,
    preprocess_dataframe,
    split_and_save_data,
)

pd.set_option('display.max_colwidth', None)
print('Импорт прошёл успешно!')
print('Pandas:', pd.__version__)

Импорт прошёл успешно!
Pandas: 3.1.1


## 1. Проверка функций на одном примере

In [2]:
test_text = ('Здравствуйте!<br>не хватило # балла по биологии. '
             'Может ли она пересдать экзамен по билогии?<br><br>С уважением, Максим.')

print('=== 1. ИСХОДНЫЙ ===')
print(test_text)

cleaned = clean_text(test_text)
print('\n=== 2. clean_text ===')
print(cleaned)

fixed = fix_typos(cleaned)
print('\n=== 3. fix_typos ===')
print(fixed)

slang = replace_slang(fixed)
print('\n=== 4. replace_slang ===')
print(slang)

lemmatized = lemmatize_text(slang)
print('\n=== 5. lemmatize_text ===')
print(lemmatized)

=== 1. ИСХОДНЫЙ ===
Здравствуйте!<br>не хватило # балла по биологии. Может ли она пересдать экзамен по билогии?<br><br>С уважением, Максим.

=== 2. clean_text ===
не хватило <num> балла по биологии может ли она пересдать экзамен по билогии

=== 3. fix_typos ===
не хватило <num> балла по биологии может ли она пересдать экзамен по биология

=== 4. replace_slang ===
не хватило <num> балла по биологии может ли она пересдать экзамен по биология

=== 5. lemmatize_text ===
хватить <num> балл биология пересдать экзамен биология


## 2. Полный пайплайн (без аугментации)

In [3]:
df = pd.read_excel('../data/mails.xlsx')
print(f'Загружено строк: {df.shape[0]}')

df_processed = preprocess_dataframe(df)

print('\n=== ПРИМЕРЫ ===')
display(df_processed[['CONTENT', 'CONTENT_clean', 'CONTENT_lemm', 'CONTENT_stem']].head(3))

print('\nРаспределение классов:')
print(df_processed['class'].value_counts().sort_index())

Загружено строк: 588
1. Применяем базовую очистку...
2. Исправляем опечатки...
3. Применяем лемматизацию...
4. Применяем стемминг (для классики)...
5. Применяем замену сленга (до лемматизации, иначе словарь не сработает)...

=== ПРИМЕРЫ ===


,CONTENT,CONTENT_clean,CONTENT_lemm,CONTENT_stem
0,"Здравствуйте! Поступила в ваш институт на бюджет Педиатрического факультета. Написала отказ от зачисления, как удостоверится в том, что меня отчислили? С уважнием, Ксения.",поступила в ваш институт на бюджет педиатрического факультета написала отказ от зачисления как удостоверится в том что меня отчислили,поступить ваш институт бюджет педиатрический факультет написать отказ зачисление удостовериться отчислить,посту ваш инсти бюдже педиа факул напис отказ зачис удост отчис
1,"Здравствуйте! Я прошла конкурс по трем дисциплинам, зачислена дисциплину по #-й приоритету, возможно ли перейти сейчас на другой приоритет(управление персоналом) или ""поезд ушел""?",я прошла конкурс по трем дисциплинам зачислена дисциплину по <num> й приоритету возможно ли перейти сейчас на другой приоритет управление персоналом или поезд ушел,пройти конкурс три дисциплина зачислить дисциплина <num> приоритет возможно перейти приоритет управление персонал поезд уйти,пройт конку три дисци зачис дисци <num> приор возмо перей приор управ персо поезд уйти
2,"Здравствуйте! Скажите пожалуйста могу ли я подать документы на направление Экология если в этом году ЕГЭ в ## классе Не сдавала есть результаты года за # класс русского языка, математики, и география.",скажите пожалуйста могу ли я подать документы на направление экология если в этом году единый государственный экзамен в <num> классе не сдавала есть результаты года за <num> класс русского языка математики и география,сказать пожалуйста мочь подать документ направление экология год единый государственный экзамен <num> класс сдавать результат год <num> класс русский язык математика география,сказа пожал мочь подат докум напра эколо год единый государственный экзамен <num> класс сдава резул год <num> класс русск язык матем геогр



Распределение классов:
class
1    289
2     24
3     29
4     28
5    155
6     42
7      5
8     16
Name: count, dtype: int64


## 3. Сплит + аугментация ТОЛЬКО train

In [ ]:
train_aug, test_df = split_and_save_data(df_processed, test_size=0.10, random_state=42, augment=True) #augment=False

print('\n=== ПРОВЕРКА СОХРАНЁННЫХ ФАЙЛОВ ===')
train_check = pd.read_parquet('../data/train.parquet')
test_check  = pd.read_parquet('../data/test.parquet')
print(f'Train: {train_check.shape[0]} строк, {train_check.shape[1]} колонок')
print(f'Test:  {test_check.shape[0]} строк, {test_check.shape[1]} колонок')
print('Колонки:', train_check.columns.tolist())

print('\nРаспределение в test:')
print(test_check['class'].value_counts().sort_index())


6. Делим данные на Train/Test (0.9/0.1)...
Выполняем аугментацию редких классов (только train)...
Медианный размер класса: 25
  Класс 7: 5 -> 25
  Класс 8: 14 -> 25
  Класс 3: уже 26 строк — пропуск
Размер train до аугментации: 529
Размер train после аугментации: 560
Train сохранен: 560 строк
Test сохранен:  59 строк

=== ПРОВЕРКА СОХРАНЁННЫХ ФАЙЛОВ ===
Train: 560 строк, 6 колонок
Test:  59 строк, 6 колонок
Колонки: ['CONTENT', 'TYPE_HOTLINE', 'class', 'CONTENT_clean', 'CONTENT_lemm', 'CONTENT_stem']

Распределение в test:
class
1    29
2     2
3     3
4     3
5    16
6     4
8     2
Name: count, dtype: int64


## 4. Метрики F1: с аугментацией vs без

Для честного сравнения нужен train **без** аугментации. Восстанавливаем его, отбрасывая дубликаты `CONTENT_lemm` (аугментированные строки — это перемешанные копии оригиналов, а не байт-в-байт дубликаты, поэтому проще пересобрать через исходный `df_processed` и тот же `random_state`).

In [5]:
from sklearn.model_selection import train_test_split

# Пересобираем train_noaug тем же сплитом, что и в split_and_save_data
train_noaug, _ = train_test_split(
    df_processed,
    test_size=0.10,
    random_state=42,
    stratify=df_processed['class'],
)
print(f'Train без аугментации: {train_noaug.shape[0]}')
print(f'Train с аугментацией:  {train_aug.shape[0]}')
print(f'Test:                  {test_df.shape[0]}')

Train без аугментации: 529
Train с аугментацией:  560
Test:                  59


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def evaluate(train, test, col):
    vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)
    Xtr = vec.fit_transform(train[col])
    Xte = vec.transform(test[col])
    clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    clf.fit(Xtr, train['class'])
    y = clf.predict(Xte)
    return (f1_score(test['class'], y, average='macro'),
            f1_score(test['class'], y, average='weighted'))

print('=== СРАВНЕНИЕ: без аугментации vs с аугментацией ===')
print(f"{'колонка':<18}{'Macro(no aug)':>14}{'Macro(aug)':>12}{'Δ':>9}{'Weighted(aug)':>15}")
for col in ['CONTENT_clean', 'CONTENT_lemm', 'CONTENT_stem']:
    m0, w0 = evaluate(train_noaug, test_df, col)
    m1, w1 = evaluate(train_aug,   test_df, col)
    print(f'{col:<18}{m0:>14.4f}{m1:>12.4f}{m1-m0:>+9.4f}{w1:>15.4f}')

=== СРАВНЕНИЕ: без аугментации vs с аугментацией ===
колонка            Macro(no aug)  Macro(aug)        Δ  Weighted(aug)
CONTENT_clean             0.8418      0.8418  +0.0000         0.8791
CONTENT_lemm              0.7963      0.7963  +0.0000         0.8744
CONTENT_stem              0.8698      0.8032  -0.0667         0.8908
